# PileFlow Generator Driver Notebook

This notebook runs the PileFlow generator from configuration to saved jet datasets.

The pipeline we are running is the following:

- configuration
- MadGraph auto-generation OR fixed LHE input
- Pythia8 showering and hadronization
- FastJet anti-kT R=0.4 clustering
- 25-feature jet table
- pileup overlay
- PUMML-style image construction
- PUPPI baseline constituent production
- saved outputs
- diagnostics and validation

The generator logic already lives in the package under: `src/pileflow_generator/`. This notebook imports and calls that existing code.

The notebook supports two starting modes:

- Mode A: fixed LHE input
    - Existing .lhe or .lhe.gz file → Pythia8 → FastJet → outputs

- Mode B: automatic MadGraph generation
    - MadGraph → generated LHE file → Pythia8 → FastJet → outputs

Both modes feed into the same downstream generator pipeline. At the end of a successful run, the generator should produce a run-specific output directory containing:

```
jets_<process>_antikt_R0.4.npy
jets_<process>_antikt_R0.4_pileup_images.npz
jets_<process>_antikt_R0.4_metadata.json
jets_<process>_antikt_R0.4_preview.txt
```

The `.npy` file contains the 25-feature per-jet table used by downstream tabular workflows. The `.npz` file contains PUMML-style image channels and constituent arrays for true, pileup-contaminated, and PUPPI-mitigated jets. The metadata and preview files document the run configuration, output schema, and a small human-readable sample of the generated table.

The notebook is allowed to control run settings such as the process name, number of events, pileup level, jet cuts, image cuts, random seeds, MadGraph command, output directory, and diagnostic options. Any change to the physics logic, reconstruction logic, image-building logic, PUPPI algorithm, or output schema should be made in the package code, not inside this notebook.

## 1. Repository and environment

This block verifies that the notebook is attached to the local PileFlow generator package.

In [ ]:
from pathlib import Path
import sys
import importlib.util

required = [
    "numpy",
    "scipy",
    "matplotlib",
    "fastjet",
    "pythia8",
    "lhapdf",
    "yaml",
    "tqdm",
    "pileflow_generator",
]

missing = []

print("\nDependency check:")
for name in required:
    found = importlib.util.find_spec(name) is not None
    print(f"{'OK' if found else 'MISSING':8s} {name}")
    if not found:
        missing.append(name)

if missing:
    raise RuntimeError(f"Missing dependencies: {missing}")

import pileflow_generator

from pileflow_generator.config import WorkflowConfig, JetConfig
from pileflow_generator.pipeline import execute_workflow
from pileflow_generator.schemas.jet_features import FEATURE_NAMES, N_FEATURES

print("\nGenerator imports:")
print("WorkflowConfig: OK")
print("JetConfig: OK")
print("execute_workflow: OK")

## 2. Package imports

This block imports the real generator objects from `src/pileflow_generator/`.

The notebook does not define its own versions of configuration classes, pipeline functions, feature schemas, image schemas, physics helpers, reconstruction stages, or diagnostic routines. All generator logic comes from the package. The most important import is `execute_workflow`, which is the production entry point used to run the full generation chain.

In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
from io import BytesIO
from IPython.display import Image, display

from pileflow_generator.config import WorkflowConfig, JetConfig
from pileflow_generator.pipeline import execute_workflow

from pileflow_generator.schemas.jet_features import (
    FEATURE_NAMES,
    N_FEATURES,
    ALGO_NAME_TO_CODE,
    ALGO_CODE_TO_NAME,
)

from pileflow_generator.schemas.image_arrays import (
    MAX_CONST,
    REQUIRED_NPZ_KEYS,
)

from pileflow_generator.diagnostics.sanity import print_header, print_sanity
from pileflow_generator.diagnostics.plots import plot_global_dataset_figures
from pileflow_generator.io.paths import ensure_dir, timestamp_string
from pileflow_generator.io.readers import decompress_lhe_if_needed
from pileflow_generator.io.writers import save_outputs

Stage-level visibility imports

In [3]:
from pileflow_generator.stages.madgraph import MadGraphRunner
from pileflow_generator.stages.pythia import PythiaRunner
from pileflow_generator.stages.clustering import FastJetRunner
from pileflow_generator.stages.pileup import PileupOverlay
from pileflow_generator.stages.images import JetImageBuilder, produce_images
from pileflow_generator.stages.baseline_puppi import run_puppi

In [ ]:
import sys

CWD = Path.cwd().resolve()

PROJECT_ROOT = None
for candidate in [CWD, *CWD.parents]:
    if (
        (candidate / "pyproject.toml").exists()
        and (candidate / "src" / "pileflow_generator").is_dir()
    ):
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not find the PileFlow generator project root. "
        "Run this notebook from inside the generator repository."
    )

SRC = PROJECT_ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC:", SRC)
print("Python executable:", sys.executable)

## 3. Outputs

Before running the generator, we first state the expected output. A successful run should create a timestamped run directory under the configured output directory:

```text
data/run_<process_name>_<timestamp>/
├── README.txt
└── antikt_R0.4/
    ├── jets_<process>_antikt_R0.4.npy
    ├── jets_<process>_antikt_R0.4_pileup_images.npz
    ├── jets_<process>_antikt_R0.4_metadata.json
    ├── jets_<process>_antikt_R0.4_preview.txt
    ├── figures/
    └── event_figures/

In [ ]:
import pandas as pd

print("N_FEATURES =", N_FEATURES)

feature_table = pd.DataFrame({
    "index": range(N_FEATURES),
    "feature": FEATURE_NAMES,
})

feature_table

### Feature groups

The 25 columns can be read in groups:

| Group | Features | Meaning |
|---|---|---|
| Generator-level kinematics | `pt_gen`, `eta_gen`, `phi_gen`, `m_gen` | Jet kinematics before detector-like smearing |
| Flavour information | `flavour` | Absolute PDG ID of the matched parton |
| Tagging proxies | `btag`, `ctag`, `qgl`, `jetId`, `nSV` | Simplified tagging and quality information |
| Reco-like kinematics | `recoPt`, `recoPhi`, `recoEta`, `recoMass` | Detector-smeared kinematic quantities |
| Constituent counts | `recoNConst`, `ncharged`, `nneutral` | Counts of visible constituents inside the jet |
| Energy fractions | `nef`, `nhf`, `cef`, `chf` | Neutral/charged electromagnetic/hadronic energy fractions |
| Bookkeeping | `muon_pT`, `jetR`, `algoCode`, `jetArea` | Auxiliary jet and algorithm metadata |

In [ ]:
required_npz_keys = list(REQUIRED_NPZ_KEYS)

npz_key_table = pd.DataFrame({
    "key": sorted(required_npz_keys)
})

print("Number of required .npz keys:", len(required_npz_keys))
print("MAX_CONST =", MAX_CONST)

npz_key_table

### `.npz` key groups

The `.npz` file contains four main groups.

#### PUMML image inputs

| Key | Shape | Meaning |
|---|---:|---|
| `ch_charged_lv` | `(N, 36, 36)` | Charged leading-vertex activity |
| `ch_charged_pu` | `(N, 36, 36)` | Charged pileup activity |
| `ch_neutral_all` | `(N, 36, 36)` | All neutral activity, LV + PU, upsampled to charged resolution |
| `ch_neutral_all_raw` | `(N, 9, 9)` | Raw neutral-all image before upsampling |

#### PUMML target and clean references

| Key | Shape | Meaning |
|---|---:|---|
| `ch_neutral_lv` | `(N, 9, 9)` | Neutral leading-vertex target |
| `clean_neutral_lv` | `(N, 9, 9)` | Clean LV neutral reference |
| `clean_neutral_all` | `(N, 9, 9)` | Clean all-neutral reference |

#### Per-image jet metadata

| Key | Shape | Meaning |
|---|---:|---|
| `jet_pt` | `(N,)` | Jet transverse momentum at image-building stage |
| `jet_eta` | `(N,)` | Jet pseudorapidity |
| `jet_phi` | `(N,)` | Jet azimuth |
| `n_pu` | `(N,)` | Actual overlaid pileup count |

#### Constituent arrays

| Key group | Shape | Meaning |
|---|---:|---|
| `true_px`, `true_py`, `true_pz`, `true_e`, `true_n` | `(N, MAX_CONST)` plus counts | LV-only constituents |
| `pileup_px`, `pileup_py`, `pileup_pz`, `pileup_e`, `pileup_n` | `(N, MAX_CONST)` plus counts | LV + PU constituents |
| `puppi_px`, `puppi_py`, `puppi_pz`, `puppi_e`, `puppi_n` | `(N, MAX_CONST)` plus counts | PUPPI-mitigated constituents |

## 4. Image-array

The file `jets_<process>_antikt_R0.4_pileup_images.npz` stores the image-level and constituent-level outputs of the generator. It is produced by the pileup/image branch of the pipeline after the hard event has been showered, clustered, and matched to selected jets.

This file is used for two related purposes:

1. PUMML-style image learning
2. True vs Pileup vs PUPPI constituent-level comparison

The image arrays describe charged and neutral transverse-momentum flow around each selected jet. The constituent arrays store zero-padded four-vector information for the LV-only, pileup-contaminated, and PUPPI-mitigated versions of the jet.

---

### Image geometry

Each image is centered on a selected jet axis.

The two image coordinates are:

- horizontal direction: $\Delta \eta$ relative to the jet axis
- vertical direction: $\Delta \phi$ relative to the jet axis

For a constituent near a selected jet, the relative coordinates are

$$
\Delta \eta = \eta_{\text{particle}} - \eta_{\text{jet}},
$$

and

$$
\Delta \phi = \phi_{\text{particle}} - \phi_{\text{jet}}.
$$

The azimuthal difference $\Delta \phi$ must be wrapped because $\phi$ is an angular coordinate. This prevents particles near $\phi = \pi$ and $\phi = -\pi$ from appearing artificially far apart.

The default image window is approximately

$$
\Delta \eta \in [-0.45, 0.45],
$$

and

$$
\Delta \phi \in [-0.45, 0.45].
$$

Therefore, the full image covers a

$$
0.9 \times 0.9
$$

patch in $(\eta,\phi)$-space around the selected jet.

The charged images use a finer grid:

$$
36 \times 36.
$$

The neutral images use a coarser grid:

$$
9 \times 9.
$$

The raw neutral image can also be upsampled from

$$
9 \times 9
$$

to

$$
36 \times 36
$$

for compatibility with the charged-image resolution.

This explains why some arrays have shape

```text
(N, 36, 36)
```

while others have shape

```text
(N, 9, 9)
```

where $N$ is the number of image-level jets stored in the `.npz` file.

---

### PUMML image inputs

The PUMML input channels are:

| Key | Shape | Meaning |
|---|---:|---|
| `ch_charged_lv` | `(N, 36, 36)` | Charged leading-vertex transverse-momentum image |
| `ch_charged_pu` | `(N, 36, 36)` | Charged pileup transverse-momentum image |
| `ch_neutral_all` | `(N, 36, 36)` | All neutral transverse momentum, LV + PU, upsampled to $36\times36$ |
| `ch_neutral_all_raw` | `(N, 9, 9)` | Same neutral-all information before upsampling |

Here:

```text
LV = leading vertex / hard-scatter event
PU = pileup from additional minimum-bias interactions
```

The important physical issue is that charged particles can be separated into LV and PU using tracking information, but neutral particles cannot be separated as directly. Therefore, the neutral input channel is contaminated:

$$
\texttt{ch\_neutral\_all}
=
\text{neutral LV}
+
\text{neutral PU}.
$$

The PUMML task is to learn the neutral leading-vertex component from the available charged and neutral information.

In other words, the model is given information similar to

$$
I_{\text{charged,LV}},
\qquad
I_{\text{charged,PU}},
\qquad
I_{\text{neutral,all}},
$$

and the desired target is

$$
I_{\text{neutral,LV}}.
$$

---

### PUMML target and clean references

The PUMML target and clean-reference arrays are:

| Key | Shape | Meaning |
|---|---:|---|
| `ch_neutral_lv` | `(N, 9, 9)` | Neutral leading-vertex target |
| `clean_neutral_lv` | `(N, 9, 9)` | Clean LV neutral reference |
| `clean_neutral_all` | `(N, 9, 9)` | Clean all-neutral reference |

The main target is

```text
ch_neutral_lv
```

because this is the neutral leading-vertex component that PUMML-style pileup mitigation tries to recover.

In the clean LV-only event, there is no pileup. Therefore,

$$
\texttt{clean\_neutral\_all}
=
\texttt{clean\_neutral\_lv}.
$$

This is expected. It is not a bug. If there is no pileup, then all neutral particles in the clean event are leading-vertex particles.

Also, overlaying pileup does not change the hard-scatter particles themselves. Therefore, the neutral leading-vertex target in the pileup-overlaid event should match the clean neutral leading-vertex reference, up to implementation details:

$$
\texttt{ch\_neutral\_lv}
\approx
\texttt{clean\_neutral\_lv}.
$$

The important pileup contamination appears in

```text
ch_neutral_all_raw
```

because that channel contains both neutral LV and neutral PU activity.

---

### Per-image jet metadata

The `.npz` file also stores one-dimensional arrays describing the jets for which images were saved.

| Key | Shape | Meaning |
|---|---:|---|
| `jet_pt` | `(N,)` | Jet transverse momentum at the image-building stage |
| `jet_eta` | `(N,)` | Jet pseudorapidity |
| `jet_phi` | `(N,)` | Jet azimuthal angle |
| `n_pu` | `(N,)` | Actual number of pileup events overlaid |

Here, $N$ is the number of jets stored in the image file.

This number does not have to be identical to the number of rows in the tabular `.npy` file. The tabular output and image output can use different cuts. For example, the tabular table is controlled by `jet_pt_min`, while the image branch can be controlled by `image_pt_min`.

---

### Constituent arrays

The `.npz` file also stores zero-padded constituent four-vectors.

There are three constituent collections:

| Collection | Meaning |
|---|---|
| `true_*` | LV-only jet constituents |
| `pileup_*` | Full pileup-contaminated jet constituents |
| `puppi_*` | PUPPI-mitigated jet constituents |

Each collection has:

```text
px
py
pz
e
n
```

Therefore, the full constituent groups are:

```text
true_px,   true_py,   true_pz,   true_e,   true_n
pileup_px, pileup_py, pileup_pz, pileup_e, pileup_n
puppi_px,  puppi_py,  puppi_pz,  puppi_e,  puppi_n
```

The four-vector arrays have shape

```text
(N, MAX_CONST)
```

and the count arrays have shape

```text
(N,)
```

The `*_n` arrays store the actual number of valid constituents before zero padding.

For example:

```text
true_n[i]
```

is the number of valid LV-only constituents stored for jet `i`, while

```text
true_px[i, :]
true_py[i, :]
true_pz[i, :]
true_e[i, :]
```

store the zero-padded four-vector components for that same jet.

Only the first `true_n[i]` entries are physical. The remaining entries are padding.

---

### True vs Pileup vs PUPPI

The constituent arrays let us compare three versions of the same jet:

```text
True:
    LV-only jet. This is the clean reference.

Pileup:
    LV + PU jet. This is the contaminated jet.

PUPPI:
    Pileup-contaminated event after applying the PUPPI particle-level algorithm.
```

The important point is that PUPPI is not a neural network. It is also not a PUMML image model. PUPPI does not run on the image tensors.

PUPPI runs on particles or constituents. It assigns weights to particles based on local event-shape information and suppresses particles that look pileup-like. The generator stores the resulting PUPPI-mitigated constituents in the `.npz` file so that later diagnostics can compare

```text
True vs Pileup vs PUPPI
```

using observables such as jet transverse momentum, jet mass, constituent multiplicity, neutral energy flow, or image-level energy sums.

---

### Summary of the `.npz` file

The image file can be understood as four groups:

| Group | Arrays | Purpose |
|---|---|---|
| PUMML inputs | `ch_charged_lv`, `ch_charged_pu`, `ch_neutral_all`, `ch_neutral_all_raw` | Inputs for image-based pileup mitigation |
| PUMML target/reference | `ch_neutral_lv`, `clean_neutral_lv`, `clean_neutral_all` | Neutral LV target and clean references |
| Jet metadata | `jet_pt`, `jet_eta`, `jet_phi`, `n_pu` | Per-image jet information |
| Constituent arrays | `true_*`, `pileup_*`, `puppi_*` | True, contaminated, and PUPPI-mitigated constituent collections |

Later in the notebook, after the generator runs, this contract will be checked by loading the `.npz` file and verifying that all required arrays exist and have the expected shapes.

## 5. Run-mode selection

This block selects how the generator obtains the hard-scatter LHE input.

There are two supported modes:

```text
fixed_lhe
    Use an existing .lhe or .lhe.gz file.
    This skips MadGraph and starts from a known LHE file.

auto_mg5
    Run MadGraph automatically from a process command.
    MadGraph generates the LHE file first, then the same downstream pipeline runs.
```

Only the LHE source changes between the two modes. After an LHE file is available, both modes run the same downstream generator pipeline:
- LHE → Pythia8 → FastJet → feature table → pileup images → PUPPI constituents → outputs

For debugging, start with `fixed_lhe`. For full generation from scratch, switch to auto_mg5.

### Select run mode

In [7]:
RUN_MODE = "auto_mg5"   # options: "fixed_lhe", "auto_mg5"

### Fixed-LHE settings

IMPORTANT: EDIT

In [8]:
# EDIT ME; i have my old LHE file in this location
REFERENCE_LHE = PROJECT_ROOT / "data" / "reference_old" / "unweighted_events_smoke_test.lhe" 

if RUN_MODE == "fixed_lhe":
    if not REFERENCE_LHE.exists():
        raise FileNotFoundError(
            "Fixed-LHE mode was selected, but the reference LHE file was not found:\n"
            f"  {REFERENCE_LHE}\n\n"
            "Either copy the reference LHE into data/reference_old/, "
            "or switch RUN_MODE to 'auto_mg5'."
        )

    print("Using fixed LHE input:")
    print(" ", REFERENCE_LHE)

### Auto-MG5 settings

In [ ]:
MG5_PROCESS_COMMAND = "generate p p > j j"
MG5_NEVENTS = 100
MG5_PTJ = 20.0
MG5_RUN_CARD_EDITS = {}

MG5_PATH_FROM_ENV = os.environ.get("MG5_PATH")

if MG5_PATH_FROM_ENV is None:
    print("MG5_PATH is not set in the environment.")
else:
    mg5_path = Path(MG5_PATH_FROM_ENV).expanduser()
    print("MG5 executable exists:", mg5_path.exists())

If `MG5_PATH is not set in the environment`, this following cell

1. checks whether mg5_aMC is already discoverable
2. falls back to ~/softwares/mg5amcnlo/bin/mg5_aMC
3. stores the path in os.environ["MG5_PATH"]

In [ ]:
import os
import shutil

# Try to find mg5_aMC from the active shell/kernel environment.
mg5_from_path = shutil.which("mg5_aMC")

print("shutil.which('mg5_aMC'):", mg5_from_path)

if mg5_from_path is not None:
    MG5_PATH = Path(mg5_from_path).resolve()
else:
    # Edit this fallback if your MadGraph install is somewhere else.
    MG5_PATH = Path("~/softwares/mg5amcnlo/bin/mg5_aMC").expanduser()

print("Candidate MG5_PATH:", MG5_PATH)
print("Exists:", MG5_PATH.exists())
print("Is file:", MG5_PATH.is_file())

if not MG5_PATH.exists():
    raise FileNotFoundError(
        "Could not find mg5_aMC.\n\n"
        "Set MG5_PATH manually to the full path of your MadGraph executable, for example:\n"
        "    MG5_PATH = Path('/path/to/mg5_aMC')\n"
    )

os.environ["MG5_PATH"] = str(MG5_PATH)

print("MG5_PATH stored in notebook environment:")
print(os.environ["MG5_PATH"])

### Derive variables for the configuration block

In [12]:
if RUN_MODE == "fixed_lhe":
    USE_MG5_AUTO = False
    LHE_FILE = REFERENCE_LHE

elif RUN_MODE == "auto_mg5":
    USE_MG5_AUTO = True
    LHE_FILE = None

else:
    raise RuntimeError("Unreachable RUN_MODE branch.")

## 6. Main run controls

This block defines the main run settings that will be passed into `WorkflowConfig` in the next block.

These variables control the size, cuts, seeds, pileup level, output location, and diagnostic behavior of the run.

### Run identity and output location

In [13]:
PROCESS_NAME = "notebook_pp_jj_test"
OUTPUT_DIR = PROJECT_ROOT / "data" / "notebook_runs"

### Event statistics

In [14]:
N_EVENTS = 10
N_PU = 1

### Physics / reconstruction cuts

In [15]:
JET_PT_MIN = 15.0
IMAGE_PT_MIN = 15.0
MIN_HARD_PARTON_PT = 0.0

### Random seeds

In [16]:
RNG_SEED = 123
PYTHIA_SEED = 123

### Diagnostics

In [17]:
SAVE_FIGURES = True
MAX_EVENT_FIGURES = 2
MAX_SCATTER_POINTS_GLOBAL = 5000

## 7. Build `WorkflowConfig`

This block converts the notebook controls into the real package configuration object.

The generator pipeline should receive one object `config`, created from: `WorkflowConfig(...)`.

In [ ]:
common_config_kwargs = {
    "process_name": PROCESS_NAME,
    "output_dir": str(OUTPUT_DIR),
    "work_dir": str(PROJECT_ROOT),
    "jet_config": JetConfig(),

    "n_events": N_EVENTS,
    "n_pu": N_PU,

    "jet_pt_min": JET_PT_MIN,
    "image_pt_min": IMAGE_PT_MIN,
    "min_hard_parton_pt": MIN_HARD_PARTON_PT,

    "rng_seed": RNG_SEED,
    "pythia_seed": PYTHIA_SEED,

    "save_figures": SAVE_FIGURES,
    "max_event_figures": MAX_EVENT_FIGURES,
    "max_scatter_points_global": MAX_SCATTER_POINTS_GLOBAL,
}

if RUN_MODE == "fixed_lhe":
    config = WorkflowConfig(
        use_mg5_auto=False,
        lhe_file=str(LHE_FILE),
        **common_config_kwargs,
    )

elif RUN_MODE == "auto_mg5":
    config = WorkflowConfig(
        use_mg5_auto=True,
        lhe_file=None,

        mg5_process_command=MG5_PROCESS_COMMAND,
        mg5_nevents=MG5_NEVENTS,
        mg5_ptj=MG5_PTJ,
        mg5_run_card_edits=MG5_RUN_CARD_EDITS,

        **common_config_kwargs,
    )

else:
    raise RuntimeError(f"Unhandled RUN_MODE: {RUN_MODE!r}")

from dataclasses import asdict

config_dict = asdict(config)

config_table = pd.DataFrame(
    [{"field": key, "value": value} for key, value in config_dict.items()]
)

config_table

## 8. Stage-by-stage full execution

This block runs the generator stage by stage.

Unlike the compact production call 

```python
result = execute_workflow(config)
```

this block manually calls the existing package stages so that each intermediate output can be inspected.

The stage sequence is:
- optional MadGraph
- LHE preparation
- Pythia8 event reading
- output-folder creation
- pileup/image-builder setup
- FastJet clustering and feature extraction
- PUMML/PUPPI image and constituent production
- output saving
- optional Feynman diagram collection

###  optional MadGraph

In [ ]:
print_header()
ensure_dir(config.output_dir)

if config.use_mg5_auto: # [0/3] Running MadGraph automatically
    mg_runner = MadGraphRunner(config.mg5_path, config.work_dir)

    mg_runner.run_automatic(
        process_name=config.process_name,
        process_command=config.mg5_process_command,
        mg5_nevents=config.mg5_nevents,
        mg5_ptj=config.mg5_ptj,
        extra_run_card_edits=config.mg5_run_card_edits,
    )

    found_lhe = mg_runner.find_lhe_file(config.process_name)

    config.lhe_file = found_lhe

    print("MadGraph LHE file:")
    print(" ", config.lhe_file)

else: # [0/3] Automatic MadGraph disabled
    print("Using existing LHE file:")
    print(" ", config.lhe_file)

### prepare LHE

In [ ]:
lhe_before = config.lhe_file
config.lhe_file = decompress_lhe_if_needed(config.lhe_file)
lhe_after = config.lhe_file

print("LHE file before preparation:")
print(" ", lhe_before)

print("\nLHE file used by Pythia:")
print(" ", lhe_after)

if lhe_before != lhe_after:
    print("\nLHE file was decompressed or redirected.")
else:
    print("\nLHE file did not require decompression.")

### Pythia

In [ ]:
# [1/3] Reading events with Pythia8
pythia_runner = PythiaRunner(
    lhe_file=config.lhe_file,
    n_events=config.n_events,
    pythia_seed=config.pythia_seed,
    min_hard_parton_pt=config.min_hard_parton_pt,
)

stored_events = pythia_runner.read_events()

In [ ]:
print(f"Accepted stored events: {len(stored_events)}")

if len(stored_events) == 0:
    raise RuntimeError(
        "Pythia returned zero stored events. "
        "The downstream FastJet and image stages will have nothing to process."
    )

first_event = stored_events[0]

print("\nType of first stored event:", type(first_event))

if hasattr(first_event, "keys"):
    print("First stored event keys:")
    print(list(first_event.keys()))

### output folders and `.npz` path

In [ ]:
ts = timestamp_string()
cfg_key = f"antikt_R{config.jet_config.R:g}"

run_dir = ensure_dir(
    os.path.join(config.output_dir, f"run_{config.process_name}_{ts}")
)

cfg_dir = ensure_dir(
    os.path.join(run_dir, cfg_key)
)

npz_name = f"jets_{config.process_name}_{cfg_key}_pileup_images.npz"
npz_path = os.path.join(cfg_dir, npz_name)

print("Timestamp:")
print(" ", ts)

print("\nConfiguration key:")
print(" ", cfg_key)

print("\nRun directory:")
print(" ", run_dir)

print("\nConfiguration output directory:")
print(" ", cfg_dir)

print("\nImage/constituent .npz path:")
print(" ", npz_path)

### pileup and image builders

In [ ]:
pileup_overlay = PileupOverlay(
    pythia_seed=config.pythia_seed + 1
)

image_builder = JetImageBuilder(
    eta_range=0.45,
    phi_range=0.45,
    n_pixels_charged=36,
    n_pixels_neutral=9,
    pt_charged_cut=0.5,
)

### FastJet clustering and image production

In [ ]:
# [2/3] Clustering jets with FastJet
fastjet_runner = FastJetRunner(
    jet_pt_min=config.jet_pt_min,
    jet_R=config.jet_config.R,
    rng_seed=config.rng_seed,
    pileup_overlay=pileup_overlay,
    image_builder=image_builder,
    n_pu=config.n_pu,
    image_pt_min=config.image_pt_min,
    image_output_path=npz_path,
)

dataset, event_figures = fastjet_runner.cluster_events(stored_events)

print(f"Jets stored in dataset: {dataset.shape[0]}")
print(f"Dataset shape: {dataset.shape}")
print(f"Number of event-figure records: {len(event_figures)}")

### save outputs

In [ ]:
# [3/3] Saving outputs
run_dir, npy_path = save_outputs(
    config=config,
    dataset=dataset,
    event_figures=event_figures,
    ts=ts,
    cfg_dir=cfg_dir,
)

print("Saved dataset file:")
print(" ", npy_path)

if os.path.exists(npy_path):
    print("Dataset .npy exists.")
else:
    raise RuntimeError(f"Expected .npy file was not created: {npy_path}")

### optional Feynman diagrams

In [ ]:
if getattr(config, "save_feynman_diagrams", False):
    print("\nCollecting MG5 Feynman diagrams...")

    mg_runner = MadGraphRunner(config.mg5_path, config.work_dir)
    mg_runner.collect_feynman_diagrams(config.process_name, run_dir)

    print("Feynman diagram collection finished.")

else:
    print("\nFeynman diagram collection disabled.")

### final summary

In [ ]:
print("\n" + "=" * 80)
print("STAGE-BY-STAGE WORKFLOW FINISHED")
print("=" * 80)
print(f"Process name : {config.process_name}")
print(f"LHE file     : {config.lhe_file}")
print(f"Output dir   : {run_dir}")
print(f"Dataset file : {npy_path}")
print(f"Image file   : {npz_path}")
print(f"N events     : {len(stored_events)}")
print(f"N jets       : {dataset.shape[0]}")
print("=" * 80)

result = (run_dir, npy_path)

output_paths = {
    "run_dir": run_dir,
    "cfg_dir": cfg_dir,
    "npy_path": npy_path,
    "npz_path": npz_path,
    "metadata_path": os.path.join(
        cfg_dir,
        f"jets_{config.process_name}_{cfg_key}_metadata.json",
    ),
    "preview_path": os.path.join(
        cfg_dir,
        f"jets_{config.process_name}_{cfg_key}_preview.txt",
    ),
    "figures_dir": os.path.join(cfg_dir, "figures"),
    "event_figures_dir": os.path.join(cfg_dir, "event_figures"),
}

pd.DataFrame(
    [
        {
            "artifact": key,
            "path": value,
            "exists": os.path.exists(value),
        }
        for key, value in output_paths.items()
    ]
)

## 9. Locate output files

This block locates the files created by the generator run.

The expected run layout is:

```text
<output_dir>/run_<process_name>_<timestamp>/
└── antikt_R0.4/
    ├── jets_<process>_antikt_R0.4.npy
    ├── jets_<process>_antikt_R0.4_pileup_images.npz
    ├── jets_<process>_antikt_R0.4_metadata.json
    ├── jets_<process>_antikt_R0.4_preview.txt
    ├── figures/
    └── event_figures/
```

This block only finds and validates paths. It does not load the .npy, .npz, metadata, or preview contents yet.

In [ ]:
cfg_key = f"antikt_R{config.jet_config.R:g}"
base_name = f"jets_{config.process_name}_{cfg_key}"

run_dir = Path(output_paths["run_dir"]).resolve()
cfg_dir = Path(output_paths["cfg_dir"]).resolve()

npy_path = cfg_dir / f"{base_name}.npy"
npz_path = cfg_dir / f"{base_name}_pileup_images.npz"
metadata_path = cfg_dir / f"{base_name}_metadata.json"
preview_path = cfg_dir / f"{base_name}_preview.txt"

figures_dir = cfg_dir / "figures"
event_figures_dir = cfg_dir / "event_figures"

output_paths = {
    "run_dir": str(run_dir),
    "cfg_dir": str(cfg_dir),
    "npy_path": str(npy_path),
    "npz_path": str(npz_path),
    "metadata_path": str(metadata_path),
    "preview_path": str(preview_path),
    "figures_dir": str(figures_dir),
    "event_figures_dir": str(event_figures_dir),
}

output_paths

### print tree

In [ ]:
print(run_dir.name + "/")
print("└── " + cfg_dir.name + "/")
print("    ├── " + npy_path.name)
print("    ├── " + npz_path.name)
print("    ├── " + metadata_path.name)
print("    ├── " + preview_path.name)

if figures_dir.exists():
    print("    ├── figures/")
else:
    print("    ├── figures/  [not found]")

if event_figures_dir.exists():
    print("    └── event_figures/")
else:
    print("    └── event_figures/  [not found]")

## 11. Metadata and preview

In [ ]:
import json

with open(Path(metadata_path), "r") as f:
    metadata = json.load(f)

preview_text = Path(preview_path).read_text()

print(f"process:    {metadata['process']}")
print(f"events:     {metadata['n_events_requested']}")
print(f"jets:       {metadata['n_jets']}")
print(f"features:   {metadata['n_features']}")
print(f"algorithm:  {metadata['algorithm']}, R={metadata['R']}")

print("\nPreview:")
print(preview_text[:3000])

## 12. Load and inspect the `.npy` table

This block loads the tabular dataset.

The `.npy` file contains one row per saved jet and one column per generator feature. The expected shape is:

```text
(N_jets, N_FEATURES)
```

IMPORTNAT ILL NEED TO CHECK THE CODE AS TO WHY `phi_gen` ISNT WRAPPED. the code still appears to work, but i need to check why

Load the `.npy` table

In [ ]:
npy_path = Path(npy_path)

jets = np.load(npy_path)

print("Loaded tabular jet dataset")
print("path:", npy_path)
print("shape:", jets.shape)
print("dtype:", jets.dtype)

Convert to pandas DataFrame

In [ ]:
df = pd.DataFrame(jets, columns=FEATURE_NAMES)

print("DataFrame shape:", df.shape)

df.head()

Summary statistics

In [ ]:
important_columns = [
    "pt_gen",
    "recoPt",
    "eta_gen",
    "recoEta",
    "phi_gen",
    "recoPhi",
    "m_gen",
    "recoMass",
    "flavour",
    "recoNConst",
    "ncharged",
    "nneutral",
    "nef",
    "nhf",
    "cef",
    "chf",
    "qgl",
    "btag",
    "ctag",
    "jetId",
]

summary_table = df[important_columns].describe().T

summary_table

Derived quantities

In [ ]:
frac_sum = df["nef"] + df["nhf"] + df["cef"] + df["chf"]
reco_gen_ratio = df["recoPt"] / np.clip(df["pt_gen"], 1e-6, None)

tabular_summary = {
    "n_jets": len(df),
    "median_pt_gen": float(df["pt_gen"].median()),
    "median_recoPt": float(df["recoPt"].median()),
    "median_reco_over_gen_pt": float(reco_gen_ratio.median()),
    "median_fraction_sum": float(frac_sum.median()),
    "fraction_sum_min": float(frac_sum.min()),
    "fraction_sum_max": float(frac_sum.max()),
}

pd.DataFrame(
    [{"quantity": key, "value": value} for key, value in tabular_summary.items()]
)

Plot tabular distributions

In [ ]:
%matplotlib inline

hist_specs = [
    ("pt_gen", "Generator-level jet $p_T$", r"$p_{T,\mathrm{gen}}$ [GeV]"),
    ("recoPt", "Reco-like jet $p_T$", r"$p_{T,\mathrm{reco}}$ [GeV]"),
    ("eta_gen", "Generator-level jet eta", r"$\eta_{\mathrm{gen}}$"),
    ("recoEta", "Reco-like jet eta", r"$\eta_{\mathrm{reco}}$"),
    ("phi_gen", "Generator-level jet phi", r"$\phi_{\mathrm{gen}}$"),
    ("recoPhi", "Reco-like jet phi", r"$\phi_{\mathrm{reco}}$"),
    ("m_gen", "Generator-level jet mass", r"$m_{\mathrm{gen}}$ [GeV]"),
    ("recoMass", "Reco-like jet mass", r"$m_{\mathrm{reco}}$ [GeV]"),
    ("qgl", "Quark/gluon likelihood proxy", "qgl"),
    ("btag", "b-tag proxy", "btag"),
    ("ctag", "c-tag proxy", "ctag"),
]

for column, title, xlabel in hist_specs:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(df[column].dropna(), bins=50)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Number of jets")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    plt.show()

Plot derived quantities

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(reco_gen_ratio[np.isfinite(reco_gen_ratio)], bins=50)
ax.set_xlabel(r"$p_{T,\mathrm{reco}} / p_{T,\mathrm{gen}}$")
ax.set_ylabel("Number of jets")
ax.set_title("Reco/gen $p_T$ ratio")
ax.grid(True, alpha=0.3)
plt.show()


fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(frac_sum[np.isfinite(frac_sum)], bins=50)
ax.set_xlabel("nef + nhf + cef + chf")
ax.set_ylabel("Number of jets")
ax.set_title("Energy-fraction sum")
ax.grid(True, alpha=0.3)
plt.show()

Plot flavour composition

In [ ]:
flavour_counts = (
    df["flavour"]
    .round()
    .astype(int)
    .value_counts()
    .sort_index()
)

fig, ax = plt.subplots(figsize=(7, 4))
flavour_counts.plot(kind="bar", ax=ax)
ax.set_xlabel("Matched flavour / PDG ID")
ax.set_ylabel("Number of jets")
ax.set_title("Matched jet flavour counts")
ax.grid(True, axis="y", alpha=0.3)
plt.show()

flavour_counts

## 14. Load and inspect the `.npz` image/constituent file

This block loads the PUMML/PUPPI-facing output file:

```text
jets_<process>_antikt_R0.4_pileup_images.npz
```

The file contains:
- PUMML image inputs
- PUMML neutral-LV target
- clean neutral references
- per-image jet metadata
- true / pileup / PUPPI constituent arrays

In [ ]:
npz_path = Path(npz_path)

images = np.load(npz_path, allow_pickle=False)
image_keys = sorted(images.files)
N_img = images["jet_pt"].shape[0]

print("Loaded image/constituent dataset")
print("path:", npz_path)
print("N_img:", N_img)
print("number of arrays:", len(image_keys))


image_shape_table = pd.DataFrame({
    "key": image_keys,
    "shape": [images[key].shape for key in image_keys],
    "dtype": [images[key].dtype for key in image_keys],
})

display(image_shape_table)


image_pt_df = pd.DataFrame({
    "charged_lv_pt": images["ch_charged_lv"].sum(axis=(1, 2)),
    "charged_pu_pt": images["ch_charged_pu"].sum(axis=(1, 2)),
    "neutral_all_raw_pt": images["ch_neutral_all_raw"].sum(axis=(1, 2)),
    "neutral_lv_target_pt": images["ch_neutral_lv"].sum(axis=(1, 2)),
    "clean_neutral_lv_pt": images["clean_neutral_lv"].sum(axis=(1, 2)),
    "clean_neutral_all_pt": images["clean_neutral_all"].sum(axis=(1, 2)),
})

display(image_pt_df.describe().T)


neutral_all_pt = images["ch_neutral_all_raw"].sum(axis=(1, 2))
neutral_lv_pt = images["ch_neutral_lv"].sum(axis=(1, 2))
neutral_pu_excess_pt = neutral_all_pt - neutral_lv_pt

neutral_pileup_summary = pd.DataFrame([
    {"quantity": "median neutral all pT", "value": float(np.median(neutral_all_pt))},
    {"quantity": "median neutral LV target pT", "value": float(np.median(neutral_lv_pt))},
    {"quantity": "median neutral PU excess pT", "value": float(np.median(neutral_pu_excess_pt))},
    {"quantity": "min neutral PU excess pT", "value": float(np.min(neutral_pu_excess_pt))},
    {"quantity": "max neutral PU excess pT", "value": float(np.max(neutral_pu_excess_pt))},
])

display(neutral_pileup_summary)


constituent_counts = pd.DataFrame({
    "true_n": images["true_n"].round().astype(int),
    "pileup_n": images["pileup_n"].round().astype(int),
    "puppi_n": images["puppi_n"].round().astype(int),
})

display(constituent_counts.describe().T)

# 15. PUMML image inspection

This block visualizes the actual image channels for one selected jet.

The PUMML input channels are:

```text
ch_charged_lv
ch_charged_pu
ch_neutral_all
```

The neutral target is: `ch_neutral_lv`. The clean references are: `clean_neutral_lv` and `clean_neutral_all`. The raw neutral-all image, `ch_neutral_all_raw`, is especially useful because it has the same $9\times9$ shape as the neutral-LV target. This makes it the right object to compare directly against `ch_neutral_lv`.

### select jet

In [ ]:
jet_idx = 0

if jet_idx < 0 or jet_idx >= N_img:
    raise IndexError(
        f"jet_idx={jet_idx} is out of range for N_img={N_img}."
    )

print("Selected jet index:", jet_idx)
print("N_img:", N_img)

### Optional interesting jet

In [ ]:
neutral_excess_per_jet = (
    images["ch_neutral_all_raw"].sum(axis=(1, 2))
    - images["ch_neutral_lv"].sum(axis=(1, 2))
)

interesting_jet_idx = int(np.argmax(neutral_excess_per_jet))

print("Jet with largest neutral excess:", interesting_jet_idx)
print("Neutral excess:", neutral_excess_per_jet[interesting_jet_idx])

# Uncomment to inspect the most pileup-contaminated neutral image.
jet_idx = interesting_jet_idx

### selected-jet metadata

In [ ]:
selected_image_metadata = {
    "jet_idx": jet_idx,
    "jet_pt": float(images["jet_pt"][jet_idx]),
    "jet_eta": float(images["jet_eta"][jet_idx]),
    "jet_phi": float(images["jet_phi"][jet_idx]),
    "n_pu": int(round(images["n_pu"][jet_idx])),
}

pd.DataFrame(
    [{"field": key, "value": value} for key, value in selected_image_metadata.items()]
)

### image sums

In [ ]:
image_sum_keys = [
    "ch_charged_lv",
    "ch_charged_pu",
    "ch_neutral_all",
    "ch_neutral_all_raw",
    "ch_neutral_lv",
    "clean_neutral_lv",
    "clean_neutral_all",
]

image_sum_rows = []

for key in image_sum_keys:
    arr = images[key][jet_idx]

    image_sum_rows.append({
        "key": key,
        "shape": arr.shape,
        "sum": float(arr.sum()),
        "max_pixel": float(arr.max()),
        "n_nonzero_pixels": int(np.count_nonzero(arr)),
    })

image_sum_table = pd.DataFrame(image_sum_rows)
image_sum_table

### simple image-sum printout

In [ ]:
for key in image_sum_keys:
    print(f"{key:24s}", images[key][jet_idx].sum())

### derived image-sum comparisons

In [ ]:
charged_total_sum = (
    images["ch_charged_lv"][jet_idx].sum()
    + images["ch_charged_pu"][jet_idx].sum()
)

neutral_all_raw_sum = images["ch_neutral_all_raw"][jet_idx].sum()
neutral_lv_sum = images["ch_neutral_lv"][jet_idx].sum()
neutral_pu_excess_sum = neutral_all_raw_sum - neutral_lv_sum

clean_neutral_lv_sum = images["clean_neutral_lv"][jet_idx].sum()
clean_neutral_all_sum = images["clean_neutral_all"][jet_idx].sum()

image_comparison_summary = {
    "charged_total_sum": float(charged_total_sum),
    "neutral_all_raw_sum": float(neutral_all_raw_sum),
    "neutral_lv_target_sum": float(neutral_lv_sum),
    "neutral_pu_excess_sum": float(neutral_pu_excess_sum),
    "clean_neutral_lv_sum": float(clean_neutral_lv_sum),
    "clean_neutral_all_sum": float(clean_neutral_all_sum),
    "clean_all_minus_clean_lv": float(clean_neutral_all_sum - clean_neutral_lv_sum),
    "target_minus_clean_lv": float(neutral_lv_sum - clean_neutral_lv_sum),
}

pd.DataFrame(
    [{"quantity": key, "value": value} for key, value in image_comparison_summary.items()]
)

### image extent

In [47]:
image_extent = [-0.45, 0.45, -0.45, 0.45]

### PUMML input images

In [ ]:
pumml_input_display = [
    ("ch_charged_lv", "Charged LV input"),
    ("ch_charged_pu", "Charged PU input"),
    ("ch_neutral_all", "Neutral all input, upsampled"),
    ("ch_neutral_all_raw", "Neutral all input, raw 9x9"),
]

for key, title in pumml_input_display:
    arr = images[key][jet_idx]

    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(
        arr,
        origin="lower",
        extent=image_extent,
        aspect="auto",
    )
    ax.set_xlabel("Delta eta")
    ax.set_ylabel("Delta phi")
    ax.set_title(f"{title}\n{key}, sum={arr.sum():.3f}")
    fig.colorbar(im, ax=ax, label="pT")
    plt.show()

### target and clean-reference images

In [ ]:
target_reference_display = [
    ("ch_neutral_lv", "Neutral LV target"),
    ("clean_neutral_lv", "Clean neutral LV reference"),
    ("clean_neutral_all", "Clean neutral all reference"),
]

for key, title in target_reference_display:
    arr = images[key][jet_idx]

    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(
        arr,
        origin="lower",
        extent=image_extent,
        aspect="auto",
    )
    ax.set_xlabel("Delta eta")
    ax.set_ylabel("Delta phi")
    ax.set_title(f"{title}\n{key}, sum={arr.sum():.3f}")
    fig.colorbar(im, ax=ax, label="pT")
    plt.show()

### neutral pileup difference image

In [ ]:
neutral_pu_image = images["ch_neutral_all_raw"][jet_idx] - images["ch_neutral_lv"][jet_idx]

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(
    neutral_pu_image,
    origin="lower",
    extent=image_extent,
    aspect="auto",
)
ax.set_xlabel("Delta eta")
ax.set_ylabel("Delta phi")
ax.set_title(
    "Neutral pileup estimate\n"
    "ch_neutral_all_raw - ch_neutral_lv, "
    f"sum={neutral_pu_image.sum():.3f}"
)
fig.colorbar(im, ax=ax, label="pT difference")
plt.show()

### neutral-difference statistics

In [ ]:
neutral_difference_checks = {
    "neutral_pu_image_sum": float(neutral_pu_image.sum()),
    "neutral_pu_image_min": float(neutral_pu_image.min()),
    "neutral_pu_image_max": float(neutral_pu_image.max()),
    "n_negative_pixels": int((neutral_pu_image < -1e-9).sum()),
    "n_positive_pixels": int((neutral_pu_image > 1e-9).sum()),
}

pd.DataFrame(
    [{"quantity": key, "value": value} for key, value in neutral_difference_checks.items()]
)

## 16. PUPPI constituent inspection

This block inspects the constituent-level arrays stored in the `.npz` file.

The three constituent collections are:

```text
true_*   = LV-only constituent jet
pileup_* = LV + PU constituent jet
puppi_*  = PUPPI-mitigated constituent jet
```

Each collection stores zero-padded four-vector arrays: `px, py, pz, e`, and a count array: `n`. The count array tells us how many entries are valid before zero padding.

### counts dataframe

In [ ]:
counts = pd.DataFrame({
    "true_n": images["true_n"].round().astype(int),
    "pileup_n": images["pileup_n"].round().astype(int),
    "puppi_n": images["puppi_n"].round().astype(int),
})

counts.describe().T

### Differences in constituent multiplicity

In [ ]:
count_relationships = pd.DataFrame({
    "pileup_minus_true": counts["pileup_n"] - counts["true_n"],
    "pileup_minus_puppi": counts["pileup_n"] - counts["puppi_n"],
    "puppi_minus_true": counts["puppi_n"] - counts["true_n"],
})

count_difference_summary = count_relationships.describe().T

count_difference_summary

### Constituent-count distributions

In [ ]:
for column in ["true_n", "pileup_n", "puppi_n"]:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(counts[column], bins=50)
    ax.set_xlabel(column)
    ax.set_ylabel("Number of jets")
    ax.set_title(f"Constituent count distribution: {column}")
    ax.grid(True, alpha=0.3)
    plt.show()

### Constituent-count difference distributions.

In [ ]:
for column in ["pileup_minus_true", "pileup_minus_puppi", "puppi_minus_true"]:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(count_relationships[column], bins=50)
    ax.set_xlabel(column)
    ax.set_ylabel("Number of jets")
    ax.set_title(f"Constituent count difference: {column}")
    ax.grid(True, alpha=0.3)
    plt.show()

### Constituent tables for the selected jet.

In [ ]:
selected_constituents = {}

for prefix in ["true", "pileup", "puppi"]:
    n_valid = int(round(images[f"{prefix}_n"][jet_idx]))

    px = images[f"{prefix}_px"][jet_idx, :n_valid]
    py = images[f"{prefix}_py"][jet_idx, :n_valid]
    pz = images[f"{prefix}_pz"][jet_idx, :n_valid]
    e = images[f"{prefix}_e"][jet_idx, :n_valid]

    pt = np.sqrt(px**2 + py**2)
    p = np.sqrt(px**2 + py**2 + pz**2)
    eta = 0.5 * np.log(
        np.clip((p + pz) / np.clip(p - pz, 1e-12, None), 1e-12, None)
    )
    phi = np.arctan2(py, px)

    selected_constituents[prefix] = pd.DataFrame({
        "px": px,
        "py": py,
        "pz": pz,
        "e": e,
        "pt": pt,
        "eta": eta,
        "phi": phi,
    })

selected_pt_summary = pd.DataFrame([
    {
        "collection": prefix,
        "n_constituents": len(table),
        "sum_constituent_pt": float(table["pt"].sum()),
        "max_constituent_pt": float(table["pt"].max()) if len(table) else 0.0,
        "mean_constituent_pt": float(table["pt"].mean()) if len(table) else 0.0,
    }
    for prefix, table in selected_constituents.items()
])

selected_pt_summary

### Reconstruct jet-level kinematics from constituent four-vectors

In [ ]:
constituent_jet_summaries = {}

for prefix in ["true", "pileup", "puppi"]:
    px_sum = images[f"{prefix}_px"].sum(axis=1)
    py_sum = images[f"{prefix}_py"].sum(axis=1)
    pz_sum = images[f"{prefix}_pz"].sum(axis=1)
    e_sum = images[f"{prefix}_e"].sum(axis=1)

    pt = np.sqrt(px_sum**2 + py_sum**2)
    mass2 = e_sum**2 - px_sum**2 - py_sum**2 - pz_sum**2
    mass = np.sqrt(np.clip(mass2, 0.0, None))

    constituent_jet_summaries[prefix] = pd.DataFrame({
        "pt": pt,
        "mass": mass,
        "energy": e_sum,
    })

constituent_kinematic_summary = pd.DataFrame([
    {
        "collection": prefix,
        "mean_pt": float(table["pt"].mean()),
        "median_pt": float(table["pt"].median()),
        "mean_mass": float(table["mass"].mean()),
        "median_mass": float(table["mass"].median()),
        "mean_energy": float(table["energy"].mean()),
        "median_energy": float(table["energy"].median()),
    }
    for prefix, table in constituent_jet_summaries.items()
])

constituent_kinematic_summary

### Compare pileup and PUPPI jet pT to truth.

In [ ]:
pileup_pt_error = (
    constituent_jet_summaries["pileup"]["pt"]
    - constituent_jet_summaries["true"]["pt"]
)

puppi_pt_error = (
    constituent_jet_summaries["puppi"]["pt"]
    - constituent_jet_summaries["true"]["pt"]
)

pt_error_summary = pd.DataFrame([
    {
        "quantity": "mean_abs_pileup_pt_minus_true",
        "value": float(np.abs(pileup_pt_error).mean()),
    },
    {
        "quantity": "mean_abs_puppi_pt_minus_true",
        "value": float(np.abs(puppi_pt_error).mean()),
    },
    {
        "quantity": "median_abs_pileup_pt_minus_true",
        "value": float(np.abs(pileup_pt_error).median()),
    },
    {
        "quantity": "median_abs_puppi_pt_minus_true",
        "value": float(np.abs(puppi_pt_error).median()),
    },
])

pt_error_summary

## 17. Built-in diagnostics

This block runs the migrated package diagnostics on the generated output files.

It has three lanes:

```text
A. Global table plots
B. PUMML image plots
C. True vs Pileup vs PUPPI quick comparison
```

The outputs are saved under: `antikt_R0.4/notebook_diagnostics/`. so that notebook diagnostics remain separate from the pipeline's automatic `figures/` and `event_figures/` folders.

### diagnostics directories

In [ ]:
diagnostics_dir = cfg_dir / "notebook_diagnostics"

global_diag_dir = diagnostics_dir / "global_table_plots"
pumml_diag_dir = diagnostics_dir / "pumml_image_plots"
quick_diag_dir = diagnostics_dir / "quick_comparison"

for directory in [
    diagnostics_dir,
    global_diag_dir,
    pumml_diag_dir,
    quick_diag_dir,
]:
    directory.mkdir(parents=True, exist_ok=True)

diagnostic_results = []

print("Diagnostics directory:")
print(" ", diagnostics_dir)

### global table plots

In [60]:
from pileflow_generator.diagnostics.plots import plot_global_dataset_figures

cfg_key = f"antikt_R{config.jet_config.R:g}"

try:
    plot_global_dataset_figures(
        dataset=jets,
        cfg_key=cfg_key,
        out_dir=str(global_diag_dir),
        max_scatter_points=config.max_scatter_points_global,
        rng=np.random.default_rng(config.rng_seed),
    )

    diagnostic_results.append({
        "diagnostic": "global table plots",
        "status": "ran",
        "output_dir": str(global_diag_dir),
        "detail": "Generated pT, eta, phi, and eta-phi overview plots.",
    })

except Exception as exc:
    diagnostic_results.append({
        "diagnostic": "global table plots",
        "status": "failed",
        "output_dir": str(global_diag_dir),
        "detail": repr(exc),
    })

    raise

### PUMML image diagnostics

In [ ]:
MAX_PUMML_MEAN_JETS = min(N_img, 1000)
MAX_PUMML_PANELS = min(N_img, 20)

pumml_clean_dir = pumml_diag_dir / "clean"
pumml_pileup_dir = pumml_diag_dir / "pileup"
pumml_compare_dir = pumml_diag_dir / "clean_vs_pileup"

for directory in [
    pumml_clean_dir,
    pumml_pileup_dir,
    pumml_compare_dir,
]:
    directory.mkdir(parents=True, exist_ok=True)

try:
    from pileflow_generator.diagnostics import image_plots

    available_image_plot_functions = [
        name for name in dir(image_plots)
        if not name.startswith("_")
    ]

    print("Available image_plots functions:")
    print(available_image_plot_functions)

    required_image_plot_functions = [
        "load_npz",
        "plot_mean_clean",
        "plot_mean_pileup",
        "plot_mean_clean_vs_pileup",
        "plot_clean",
        "plot_pileup",
        "plot_clean_vs_pileup",
    ]

    missing_image_plot_functions = [
        name for name in required_image_plot_functions
        if not hasattr(image_plots, name)
    ]

    if missing_image_plot_functions:
        raise AttributeError(
            "image_plots module is missing expected functions:\n"
            + "\n".join(f"  - {name}" for name in missing_image_plot_functions)
        )

    imgs = image_plots.load_npz(str(npz_path))

    image_plots.plot_mean_clean(
        imgs,
        MAX_PUMML_MEAN_JETS,
        save_path=str(pumml_clean_dir / "mean.png"),
    )

    image_plots.plot_mean_pileup(
        imgs,
        MAX_PUMML_MEAN_JETS,
        save_path=str(pumml_pileup_dir / "mean.png"),
    )

    image_plots.plot_mean_clean_vs_pileup(
        imgs,
        MAX_PUMML_MEAN_JETS,
        save_path=str(pumml_compare_dir / "mean.png"),
    )

    for i in range(MAX_PUMML_PANELS):
        image_plots.plot_clean(
            imgs,
            i,
            save_path=str(pumml_clean_dir / f"jet_{i:04d}.png"),
        )

        image_plots.plot_pileup(
            imgs,
            i,
            save_path=str(pumml_pileup_dir / f"jet_{i:04d}.png"),
        )

        image_plots.plot_clean_vs_pileup(
            imgs,
            i,
            save_path=str(pumml_compare_dir / f"jet_{i:04d}.png"),
        )

    diagnostic_results.append({
        "diagnostic": "PUMML image plots",
        "status": "ran",
        "output_dir": str(pumml_diag_dir),
        "detail": (
            f"Saved mean panels using {MAX_PUMML_MEAN_JETS} jets and "
            f"per-jet panels for {MAX_PUMML_PANELS} jets."
        ),
    })

except Exception as exc:
    diagnostic_results.append({
        "diagnostic": "PUMML image plots",
        "status": "failed",
        "output_dir": str(pumml_diag_dir),
        "detail": repr(exc),
    })

    print("PUMML image diagnostics failed:")
    print(repr(exc))

### quick comparison

In [ ]:
MAX_QUICK_COMPARISON_JETS = min(N_img, 5000)

try:
    from pileflow_generator.diagnostics import quick_comparison

    available_quick_functions = [
        name for name in dir(quick_comparison)
        if not name.startswith("_")
    ]

    print("Available quick_comparison functions:")
    print(available_quick_functions)

    required_quick_functions = [
        "collect",
        "make_plot",
    ]

    missing_quick_functions = [
        name for name in required_quick_functions
        if not hasattr(quick_comparison, name)
    ]

    if missing_quick_functions:
        raise AttributeError(
            "quick_comparison module is missing expected functions:\n"
            + "\n".join(f"  - {name}" for name in missing_quick_functions)
        )

    quick_store, quick_mean_npu, quick_n_jets = quick_comparison.collect(
        str(npz_path),
        max_jets=MAX_QUICK_COMPARISON_JETS,
    )

    quick_png = quick_comparison.make_plot(
        quick_store,
        quick_mean_npu,
        quick_n_jets,
        str(quick_diag_dir),
        process=config.process_name,
    )

    diagnostic_results.append({
        "diagnostic": "True vs Pileup vs PUPPI quick comparison",
        "status": "ran",
        "output_dir": str(quick_diag_dir),
        "detail": f"Processed {quick_n_jets} jets. Main plot: {quick_png}",
    })

except Exception as exc:
    diagnostic_results.append({
        "diagnostic": "True vs Pileup vs PUPPI quick comparison",
        "status": "failed",
        "output_dir": str(quick_diag_dir),
        "detail": repr(exc),
    })

    print("Quick comparison diagnostic failed:")
    print(repr(exc))

### collect all generated diagnostic files

In [ ]:
diagnostic_file_rows = []

for path in sorted(diagnostics_dir.rglob("*")):
    if path.is_file():
        diagnostic_file_rows.append({
            "relative_path": str(path.relative_to(diagnostics_dir)),
            "full_path": str(path),
            "size_MB": path.stat().st_size / 1024**2,
        })

diagnostic_file_table = pd.DataFrame(diagnostic_file_rows)
diagnostic_file_table

### results table

In [ ]:
diagnostic_results_table = pd.DataFrame(diagnostic_results)
diagnostic_results_table

### display of selected diagnostic images

In [ ]:
from IPython.display import Image, display

preview_diagnostic_files = [
    global_diag_dir / "hist_pt.png",
    global_diag_dir / "scatter_eta_phi_ptcolor.png",
    pumml_clean_dir / "mean.png",
    pumml_pileup_dir / "mean.png",
    pumml_compare_dir / "mean.png",
    quick_diag_dir / "quick_comparison.png",
]

for path in preview_diagnostic_files:
    if path.exists():
        print(path.relative_to(diagnostics_dir))
        display(Image(filename=str(path)))

## Final interpretation

This run generated both main generator outputs.

The `.npy` file is the tabular jet dataset. It contains one row per saved jet and 25 generator features per row.

The `.npz` file is the PUMML/PUPPI-facing dataset. It contains PUMML-style image channels, neutral-LV targets, clean neutral references, and true/pileup/PUPPI constituent arrays.

Because this notebook executed the workflow stage by stage, it also exposed the intermediate objects:

```text
stored_events
dataset
event_figures
images
df
selected_constituents
```

### final report

In [ ]:
report = {
    "run_dir": str(run_dir),
    "cfg_dir": str(cfg_dir),
    "npy_path": str(npy_path),
    "npz_path": str(npz_path),
    "metadata_path": str(metadata_path),
    "preview_path": str(preview_path),

    "n_jets_table": int(jets.shape[0]),
    "n_features": int(jets.shape[1]),
    "expected_n_features": int(N_FEATURES),
    "n_image_jets": int(N_img),

    "run_mode": RUN_MODE,
    "process_name": config.process_name,
    "n_events": int(config.n_events),
    "n_pu": int(config.n_pu),
    "jet_pt_min": float(config.jet_pt_min),
    "image_pt_min": float(config.image_pt_min),
    "min_hard_parton_pt": float(config.min_hard_parton_pt),
    "rng_seed": int(config.rng_seed),
    "pythia_seed": int(config.pythia_seed),

    "jet_algorithm": config.jet_config.algo_name,
    "jet_R": float(config.jet_config.R),
    "jet_algo_code": int(config.jet_config.algo_code),

    "execution_style": "stage-by-stage notebook execution",
}

if "diagnostics_dir" in globals():
    report["diagnostics_dir"] = str(diagnostics_dir)

if "diagnostic_file_table" in globals():
    report["n_diagnostic_files"] = int(len(diagnostic_file_table))

if "diagnostic_results_table" in globals():
    report["n_diagnostics_requested"] = int(len(diagnostic_results_table))
    report["n_diagnostics_failed"] = int(
        (diagnostic_results_table["status"] == "failed").sum()
    )

if "metadata" in globals():
    report["metadata_process"] = metadata.get("process")
    report["metadata_n_jets"] = metadata.get("n_jets")
    report["metadata_n_features"] = metadata.get("n_features")
    report["metadata_algorithm"] = metadata.get("algorithm")
    report["metadata_R"] = metadata.get("R")

report_table = pd.DataFrame(
    [{"field": key, "value": value} for key, value in report.items()]
)

report_table

### optional saved report

In [ ]:
import json

if "diagnostics_dir" in globals():
    notebook_report_path = Path(diagnostics_dir) / "notebook_run_report.json"
else:
    notebook_report_path = Path(cfg_dir) / "notebook_run_report.json"

with open(notebook_report_path, "w") as f:
    json.dump(report, f, indent=2)

print("Saved notebook run report:")
print(" ", notebook_report_path)